# Search result filtering.
The purpose of this notebook is to get a way to condense search results into a sensible output.

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda
from utils import *
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage


In [4]:
prompt_text = open_file('../prompts/SelectWebResult1.txt')
human = "{text}"
prompt = ChatPromptTemplate.from_messages([("system", prompt_text), ("human", human)])
output_parser = JsonOutputParser()
model = ChatGroq(model_name='llama3-70b-8192')
chain = prompt | model | RunnableLambda(FilterOutExtraToJSON) | output_parser


In [7]:
search_str = 'weather in Sydney today'
search = TavilySearchResults()
search_results = search.invoke({'query': search_str})

Now looking at the documentation for langchain it would appear that I have been doing things wrong. Lets see if the pydantic stuff works with our grok model.

In [9]:
import json
from typing import List

from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field

In [10]:
class ToolToCall(BaseModel):
    """Information about which tool to call."""

    thought: str = Field(..., description="step by step thinking to identify the additional information required to respond to the user.")
    search: str = Field(..., description="search terms that should be used by any function that needs them.")
    function: str = Field(..., description="The function to be used to gain the additional information required to answer the user. It should contain one value from the list: ['read clipboard', 'take screenshot', 'capture webcam', 'search web', 'None'].")